# Notebook 03 — Data Cleaning Rules Agent (Layer 2, Agent 2)

**Abstract:** Evaluates the LLM's ability to automatically detect and apply
data cleaning rules. Measures cleaning recall and data quality
improvement.

**References:**
- Annam (2025) — 'LLMs can automatically propose cleansing rules by analyzing data samples and metadata.'
- El-Sappagh et al. (2011) — Data warehouse ETL processes
- Zhang et al. (2024) — DVR self-correction framework

In [ ]:
# Install required packages into the active kernel environment (run once)
%pip install -q lxml matplotlib seaborn pandas numpy scipy pydantic requests tqdm httpx faiss-cpu sentence-transformers google-generativeai


In [1]:
# Cell 1 — Setup
%matplotlib inline
import sys, os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

def _find_research_root() -> str:
    sentinel = "generate_datasets.py"
    candidate = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.exists(os.path.join(candidate, sentinel)):
            return candidate
        parent = os.path.dirname(candidate)
        if parent == candidate:
            break
        candidate = parent
    sub = os.path.join(os.path.abspath(os.getcwd()), "research")
    if os.path.exists(os.path.join(sub, sentinel)):
        return sub
    raise FileNotFoundError(
        f"Could not locate research root (sentinel '{sentinel}' not found). "
        "Run from bi-platform/ or bi-platform/research/."
    )

RESEARCH_ROOT = _find_research_root()
os.chdir(RESEARCH_ROOT)
if RESEARCH_ROOT not in sys.path:
    sys.path.insert(0, RESEARCH_ROOT)
print(f"RESEARCH_ROOT = {RESEARCH_ROOT}")

from src.ingestion import MultiSourceIngester
from src.profiler import SchemaProfiler
from src.llm_client import MockLLMClient, LLMClient
from src.cleaning_agent import CleaningAgent
from src.evaluator import ETLEvaluator
from src.visualizer import ResearchVisualizer
from data.ground_truth.ground_truth import GROUND_TRUTH

ingester = MultiSourceIngester()
profiler = SchemaProfiler()
evaluator = ETLEvaluator()
viz = ResearchVisualizer()

try:
    real_client = LLMClient()
    llm_client = real_client if real_client.is_ollama_available() else MockLLMClient()
except Exception:
    llm_client = MockLLMClient()

cleaner = CleaningAgent(llm_client=llm_client)
datasets = ingester.ingest_all()
print(f'Loaded {len(datasets)} datasets, LLM: {type(llm_client).__name__}')


Loaded 4 datasets, LLM: LLMClient


In [2]:
# Cell 2 — Data quality baseline
gt_keys = {
    'dataset1_retail_sales': 'dataset1_retail_sales',
    'dataset2_hospital_records': 'dataset2_hospital_records',
    'dataset3_supplier_invoices': 'dataset3_supplier_invoices',
    'dataset4_ecommerce_events': 'dataset4_ecommerce_events',
}

dq_before = {}
for fname, df in datasets.items():
    short = fname.split('.')[0]
    score = evaluator.compute_dq_score(df)
    dq_before[short] = score
    print(f'{short}: DQ score = {score:.4f}')

dataset1_retail_sales: DQ score = 0.9942
dataset2_hospital_records: DQ score = 0.9920
dataset3_supplier_invoices: DQ score = 0.9967
dataset4_ecommerce_events: DQ score = 0.6578


In [3]:
# Cell 3 — Run cleaning agent on all 4 datasets
import random
random.seed(42)

cleaning_plans = {}
cleaning_recall_scores = {}

for fname, df in datasets.items():
    short = fname.split('.')[0]
    gt_key = gt_keys.get(short)
    if not gt_key or gt_key not in GROUND_TRUTH:
        continue
    
    gt = GROUND_TRUTH[gt_key]
    ctx = profiler.profile(df, short)
    
    # Detect cleaning rules
    plan = cleaner.detect_rules(ctx, df)
    cleaning_plans[short] = plan
    
    print(f"\n{'='*60}")
    print(f"Dataset: {short} (confidence: {plan.confidence:.2f})")
    print(f"Detected {len(plan.rules)} rules:")
    for rule in plan.rules:
        print(f"  [{rule.priority}] {rule.rule_type}:{rule.target_column} — {rule.description}")
    
    # Compute cleaning recall
    detected_rules = [f"{r.rule_type}:{r.target_column}" for r in plan.rules]
    expected_rules = gt['expected_cleaning_rules']
    recall = evaluator.cleaning_recall(detected_rules, expected_rules)
    cleaning_recall_scores[short] = recall
    
    print(f"\nExpected rules: {expected_rules}")
    print(f"Detected rules: {detected_rules}")
    print(f"Cleaning recall: {recall:.2f}")

LLaMA call failed: Server error '500 Internal Server Error' for url 'http://localhost:11434/api/generate'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
No ANTHROPIC_API_KEY set, Claude unavailable



Dataset: dataset1_retail_sales (confidence: 0.00)
Detected 0 rules:

Expected rules: ['standardize_date_format:order_date', 'fill_null:discount_pct:0', 'normalize_text:customer_country', 'remove_negative:quantity', 'strip_currency_symbol:unit_price', 'drop_duplicates:order_id']
Detected rules: []
Cleaning recall: 0.00


LLaMA call failed: Server error '500 Internal Server Error' for url 'http://localhost:11434/api/generate'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
No ANTHROPIC_API_KEY set, Claude unavailable



Dataset: dataset2_hospital_records (confidence: 0.00)
Detected 0 rules:

Expected rules: ['normalize_text:patient.gender', 'fill_null:discharge_date:still_admitted', 'fix_inconsistency:total_cost', 'normalize_text:diagnosis.severity']
Detected rules: []
Cleaning recall: 0.00


LLaMA call failed: Server error '500 Internal Server Error' for url 'http://localhost:11434/api/generate'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
No ANTHROPIC_API_KEY set, Claude unavailable



Dataset: dataset3_supplier_invoices (confidence: 0.00)
Detected 0 rules:

Expected rules: ['normalize_text:status', 'fix_vat_computation', 'standardize_date_format:paid_on', 'drop_duplicates:invoice_id']
Detected rules: []
Cleaning recall: 0.00


TypeError: unhashable type: 'list'

In [ ]:
# Cell 4 — Apply cleaning rules and measure improvement
dq_after = {}
cleaned_datasets = {}

for short, plan in cleaning_plans.items():
    # Find original df
    orig_df = None
    for fname, df in datasets.items():
        if fname.split('.')[0] == short:
            orig_df = df
            break
    
    if orig_df is None:
        continue
    
    # Apply cleaning
    df_clean = cleaner.apply_rules(orig_df, plan)
    cleaned_datasets[short] = df_clean
    
    score_after = evaluator.compute_dq_score(df_clean)
    dq_after[short] = score_after
    improvement = evaluator.compute_dq_improvement(orig_df, df_clean)
    
    print(f"\n{short}:")
    print(f"  Nulls before: {orig_df.isnull().sum().sum()}, after: {df_clean.isnull().sum().sum()}")
    print(f"  Rows before: {len(orig_df)}, after: {len(df_clean)}")
    print(f"  DQ score: {dq_before[short]:.4f} → {score_after:.4f}")
    print(f"  Improvement: {improvement:+.2%}")

In [ ]:
# Cell 5 — DQ improvement visualization
ds_names = list(dq_before.keys())
before_scores = [dq_before[n] for n in ds_names]
after_scores = [dq_after.get(n, dq_before[n]) for n in ds_names]

fig = viz.plot_dq_improvement(before_scores, after_scores, ds_names)
display(fig); plt.close(fig)
print('Saved: results/figures/fig4_dq_improvement.pdf')

In [ ]:
# Cell 6 — Rule detection analysis
print('=== Rule Detection Analysis ===')

rule_types = set()
for gt_key, gt_val in GROUND_TRUTH.items():
    for rule in gt_val['expected_cleaning_rules']:
        rule_types.add(rule.split(':')[0])

detection_matrix = []
for rule_type in sorted(rule_types):
    row = {'Rule Type': rule_type}
    detected_count = 0
    total_count = 0
    for short, plan in cleaning_plans.items():
        gt_key = gt_keys.get(short)
        if not gt_key:
            continue
        gt = GROUND_TRUTH[gt_key]
        expected_of_type = [r for r in gt['expected_cleaning_rules'] if r.startswith(rule_type)]
        detected_of_type = [r for r in plan.rules if r.rule_type == rule_type]
        total_count += len(expected_of_type)
        detected_count += min(len(detected_of_type), len(expected_of_type))
    
    rate = detected_count / total_count if total_count > 0 else 0
    row['Detection Rate'] = f'{rate:.0%}'
    row['Detected'] = detected_count
    row['Expected'] = total_count
    detection_matrix.append(row)

det_df = pd.DataFrame(detection_matrix)
display(det_df)

In [ ]:
# Cell 7 — Save metrics
os.makedirs('results/metrics', exist_ok=True)

metrics = {
    'cleaning_recall_per_dataset': cleaning_recall_scores,
    'dq_score_before': dq_before,
    'dq_score_after': dq_after,
    'dq_improvement_pct': {
        ds: ((dq_after.get(ds, dq_before[ds]) / dq_before[ds]) - 1) * 100
        for ds in dq_before
    },
    'rules_detected': {
        ds: [f"{r.rule_type}:{r.target_column}" for r in plan.rules]
        for ds, plan in cleaning_plans.items()
    },
    'rules_missed': {
        ds: [
            r for r in GROUND_TRUTH[gt_keys[ds]]['expected_cleaning_rules']
            if not any(
                r.split(':')[0] == det.rule_type
                for det in cleaning_plans[ds].rules
            )
        ]
        for ds in cleaning_plans if ds in gt_keys and gt_keys[ds] in GROUND_TRUTH
    },
}

with open('results/metrics/03_cleaning.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved: results/metrics/03_cleaning.json')
print('\n✓ Notebook 03 complete.')